# Player Experience Modeling — Predictive Modeling Exercise (Lecturer Version)

This notebook mirrors the student version, but includes example implementations and talking points for class demonstration.

## The Prediction Problem

We want to estimate a hidden variable:

**Player Skill Tier**

Possible outputs:
- Bronze
- Silver
- Gold
- Diamond

### Inputs

We do not observe skill directly.

We observe telemetry such as:
- KDR
- Captures
- Returns
- Interceptions
- TimeNearCarrier
- DefenseStopsNearFlag
- etc.

### Framing

Input: player telemetry  
Output: predicted skill tier

This is a **supervised classification problem**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

df = pd.read_csv("students_dataset.csv")
truth = pd.read_csv("ground_truth_hidden.csv")

print("students_dataset shape:", df.shape)
print("ground_truth_hidden shape:", truth.shape)

df.head()

## 1. Inspect the dataset

In [ ]:
print(df.columns.tolist())
print()
df.describe().T[["mean", "std", "min", "max"]].round(3)

### Talking point

At this stage, remind students that some variables reflect:
- mechanical ability
- objective contribution
- support / defense role behavior
- normalized or per-match versions of the same signal

## Descriptive vs Predictive Modeling

Last lecture: **descriptive modeling**
- understand patterns
- summarize what happened
- visualize distributions and relationships

This lecture: **predictive modeling**
- estimate hidden outcomes
- learn from labeled examples
- evaluate on unseen data

Key distinction:

Descriptive → explains the data  
Predictive → estimates unknown targets

## 2. Short Exercise — Is this predictive?

In [ ]:
df.groupby("PreferredRole")[[
    "CapturesPerMatch",
    "TimeNearCarrierPerMatch",
    "DefenseStopsNearFlagPerMatch"
]].mean().round(3)

### Talking point

This is descriptive, not predictive.

It tells us how roles differ on average, but it does not yet predict the skill of a specific individual.

## 3. Visual exploration

In [ ]:
plt.figure()
df["KDR"].hist(bins=20)
plt.title("Distribution of KDR")
plt.xlabel("KDR")
plt.ylabel("Count")
plt.show()

plt.figure()
plt.scatter(df["CapturesPerMatch"], df["WinRate"])
plt.title("CapturesPerMatch vs WinRate")
plt.xlabel("CapturesPerMatch")
plt.ylabel("WinRate")
plt.show()

plt.figure()
plt.scatter(df["TimeNearCarrierPerMatch"], df["KillsNearCarrierPerMatch"])
plt.title("TimeNearCarrierPerMatch vs KillsNearCarrierPerMatch")
plt.xlabel("TimeNearCarrierPerMatch")
plt.ylabel("KillsNearCarrierPerMatch")
plt.show()

## 4. Feature engineering

In [ ]:
df["ObjectiveImpact"] = (
    df["CapturesPerMatch"]
    + df["ReturnsPerMatch"]
    + df["InterceptionsPerMatch"]
)

df["SupportImpact"] = (
    df["TimeNearCarrierPerMatch"]
    + df["KillsNearCarrierPerMatch"]
    + df["KillsWhileCarrierAlivePerMatch"]
)

df["DefenseImpact"] = (
    df["DefenseStopsNearFlagPerMatch"]
    + df["FlagRoomPresenceUnderThreatPerMatch"]
    + df["ReturnsUnderPressurePerMatch"]
)

df["RiskScore"] = df["Overextensions"] / df["Matches"]

df[[
    "PlayerName",
    "ObjectiveImpact",
    "SupportImpact",
    "DefenseImpact",
    "RiskScore"
]].head()

### Talking point

This is a good moment to explain that:
- runners should shine more in objective metrics
- supports in carrier-adjacent metrics
- defenders in base-defense metrics
- overextensions may indicate undisciplined play

## 5. Baseline ranking model

In [ ]:
df["BaselineScore"] = (
    0.30 * df["ObjectiveImpact"]
    + 0.20 * df["SupportImpact"]
    + 0.20 * df["DefenseImpact"]
    + 0.15 * df["WinRate"]
    + 0.10 * df["KDR"]
    - 0.05 * df["RiskScore"]
)

df["BaselinePercentile"] = df["BaselineScore"].rank(pct=True)

def percentile_to_tier(p):
    if p <= 0.40:
        return "Bronze"
    elif p <= 0.75:
        return "Silver"
    elif p <= 0.95:
        return "Gold"
    return "Diamond"

df["BaselineTier"] = df["BaselinePercentile"].apply(percentile_to_tier)

df[["PlayerID", "PlayerName", "BaselineScore", "BaselineTier"]].sort_values(
    "BaselineScore", ascending=False
).head(10)

## Baseline Model vs Machine Learning

The baseline is valuable because it is:
- transparent
- easy to critique
- easy to improve

Now we ask:

**Does machine learning outperform this baseline?**

## 6. Reveal hidden ground truth and evaluate the baseline

In [ ]:
full = df.merge(truth[["PlayerID", "TrueTier"]], on="PlayerID", how="inner")

baseline_acc = accuracy_score(full["TrueTier"], full["BaselineTier"])
print("Baseline tier accuracy:", round(baseline_acc, 3))

pd.crosstab(full["TrueTier"], full["BaselineTier"], rownames=["True"], colnames=["Predicted"])

## 7. Train vs Test Data

To evaluate predictive models properly, we must separate:
- training data
- test data

Otherwise the model may memorize rather than generalize.

Training set → learn patterns  
Test set → predict unseen examples

## 8. Supervised model example: Random Forest

In [ ]:
feature_cols = [
    "KDR",
    "WinRate",
    "CapturesPerMatch",
    "ReturnsPerMatch",
    "InterceptionsPerMatch",
    "ObjectiveActionsPerMatch",
    "TimeNearCarrierPerMatch",
    "KillsNearCarrierPerMatch",
    "KillsWhileCarrierAlivePerMatch",
    "DefenseStopsNearFlagPerMatch",
    "FlagRoomPresenceUnderThreatPerMatch",
    "ReturnsUnderPressurePerMatch",
    "RiskScore",
    "ObjectiveImpact",
    "SupportImpact",
    "DefenseImpact"
]

X = full[feature_cols]
y = full["TrueTier"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Random Forest Accuracy:", round(accuracy_score(y_test, rf_pred), 3))
print()
print(classification_report(y_test, rf_pred))

## 9. Alternative example: Logistic Regression

In [ ]:
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])

logreg.fit(X_train, y_train)
lr_pred = logreg.predict(X_test)

print("Logistic Regression Accuracy:", round(accuracy_score(y_test, lr_pred), 3))
print()
print(classification_report(y_test, lr_pred))

### Talking point

Good comparison:
- Logistic Regression → simpler, linear, more interpretable
- Random Forest → nonlinear, often stronger, easier feature importance

## 10. Compare baseline vs ML

In [ ]:
print("Baseline accuracy:", round(baseline_acc, 3))
print("Random Forest accuracy:", round(accuracy_score(y_test, rf_pred), 3))
print("Logistic Regression accuracy:", round(accuracy_score(y_test, lr_pred), 3))

## 11. Feature importance

In [ ]:
fi = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi.round(4))

plt.figure()
fi.sort_values().plot(kind="barh")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.show()

### Talking points

Ask the class:
- Are the most important features what we expected?
- Do role-aware signals help compared to pure combat?
- Does the model reward objective play?
- Is KDR alone enough?

## 12. Reflection

Player skill is a **latent variable**.

We never observe skill directly.

We infer it from behavioral signals such as:
- combat performance
- objective play
- support behavior
- defensive discipline

This is the core idea of predictive player modeling.